# `StructurePreprocessor.encode_pdb()`

`encode_pdb` extracts per-residue features straight from PDB / CIF ATOM records — AlphaFold model-file features (`plddt`, `plddt_disorder`, `plddt_tier`, sidechain `chi*_sincos`, `ca_centroid_dist*`, `contact_count_*`, `hse`) and experimental `bfactor` / `depth` — into a `[0, 1]`-normalized `dict_num`. Here we use the bundled `AF_TINY` fixture (`depth` is omitted as it needs the external `msms` binary).

In [1]:
import warnings
from pathlib import Path
import numpy as np
import pandas as pd
import aaanalysis as aa
import aaanalysis.utils as ut
aa.options['verbose'] = False
warnings.filterwarnings('ignore')

PDB_FIXTURES = Path(aa.__file__).resolve().parent / '_data' / 'pdb_test'
strp = aa.StructurePreprocessor(verbose=False)
df_seq = pd.DataFrame({'entry': ['AF_TINY'],
                       'sequence': ['ACDEFGHIKLMNPQRSTVWYACDEFGHIKL']})

feats = ['plddt', 'plddt_disorder', 'plddt_tier',
         'contact_count_8A', 'bfactor']
dict_pdb = strp.encode_pdb(df_seq=df_seq, pdb_folder=str(PDB_FIXTURES),
                          features=feats)
arr = dict_pdb['AF_TINY']
print('shape (L, D):', arr.shape)
print('value range:', round(float(np.nanmin(arr)), 3),
      '..', round(float(np.nanmax(arr)), 3))

shape (L, D): (30, 8)
value range: 0.0 .. 1.0


Each row is a residue, each column a feature dimension in `[0, 1]`. Pass `return_df=True` for the `(dict_num, df_seq_out)` form whose `pdb_ok` column flags entries whose structure file failed to load.

In [2]:
# Further parameters: ``plddt_disorder_threshold`` sets the pLDDT below which a
# residue counts as disordered, ``on_failure`` governs unreadable files, and
# ``return_df=True`` also returns a per-row status frame (``pdb_ok``).
dict_pdb_thr, df_pdb_status = strp.encode_pdb(
    df_seq=df_seq, pdb_folder=str(PDB_FIXTURES),
    features=['plddt', 'plddt_disorder'],
    plddt_disorder_threshold=50.0, on_failure='nan', return_df=True)
aa.display_df(df_pdb_status, n_rows=10, show_shape=True)

DataFrame shape: (1, 3)


,entry,sequence,pdb_ok
1,AF_TINY,ACDEFGHIKLMNPQRSTVWYACDEFGHIKL,True
